In [1]:
import flax.linen as nn
import numpy as np
import optax
import jax.numpy as jnp
import swarmrl as srl
from swarmrl.models.interaction_model import Action
from swarmrl.observables import Observable
import jax
import h5py as hf
import matplotlib.pyplot as plt

import swarmrl.engine.espresso as espresso
from swarmrl.utils import utils

import pint
from typing import List

## Simulation

In [2]:
simulation_name = "min-finding"
seed = np.random.randint(0, 3453276453)
temperature = 150.0
n_colloids = 200

ureg = pint.UnitRegistry()
md_params = espresso.MDParams(
    ureg=ureg,
    fluid_dyn_viscosity=ureg.Quantity(8.9e-4, "pascal * second"),
    WCA_epsilon=ureg.Quantity(273.15, "kelvin") * ureg.boltzmann_constant,
    temperature=ureg.Quantity(temperature, "kelvin"),
    box_length=ureg.Quantity(1000, "micrometer"),
    time_slice=ureg.Quantity(1.0, "second"),  # model timestep
    time_step=ureg.Quantity(0.01, "second"),  # integrator timestep
    write_interval=ureg.Quantity(2, "second"),
)

system_runner = srl.espresso.EspressoMD(
    md_params=md_params,
    n_dims=2,
    seed=seed,
    out_folder='./quadrouple-well',
    write_chunk_size=100,
)

system_runner.add_colloids(
    n_colloids,
    ureg.Quantity(2.14, "micrometer"),
    ureg.Quantity(np.array([500, 500, 0]), "micrometer"),
    ureg.Quantity(400, "micrometer"),
    type_colloid=0,
)

In [3]:
n_slices = int(ureg.Quantity(3, "minute") / md_params.time_slice)
n_episodes = 300
episode_length = 50 #int(np.ceil(n_slices / 10))

## Reinforcement Learning

In [4]:
# Exploration policy
exploration_policy = srl.exploration_policies.RandomExploration(probability=0.2)

# Sampling strategy
sampling_strategy = srl.sampling_strategies.GumbelDistribution()

# Value function
value_function = srl.value_functions.ExpectedReturns(
    gamma=0.99, standardize=True
)


In [5]:
class MinimaDetection(srl.tasks.Task):
    """
    Task for minima detection.
    """
    def __init__(
        self,
        potential_fn: callable,
        particle_type: int = 0,
    ):
        """
        Constructor for the find origin task.

        Parameters
        ----------
        source : np.ndarray (default = (0, 0 0))
                Source of the gradient.
        potenial_fn : callable
                Function we are moving along.
        particle_type : int (default=0)

        """
        super().__init__(particle_type=particle_type)
        self.potential_fn = potential_fn

        # Class only attributes
        self._historic_potential = {}
        
    def initialize(self, colloids: List):
        """
        Prepare the task for running.

        Parameters
        ----------
        colloids : List[Colloid]
                List of colloids to be used in the task.

        Returns
        -------
        observable :
                Returns the observable required for the task.
        """
        for item in colloids:
            if item.type == self.particle_type:
                index = item.id
                function_value = self.potential_fn(item.pos)
                self._historic_potential[str(index)] = function_value
                
    def compute_colloid_reward(self, index: int, colloids):
        """
        Compute the reward for a single colloid.

        Parameters
        ----------
        index : int
                Index of the colloid to compute the reward for.
        colloids : list
                All colloids

        Returns
        -------
        reward : float
                Reward for the colloid.
        """
        colloid_id = colloids[index].id
        # Get the current position of the colloid
        current_position =colloids[index].pos
        current_function_value = self.potential_fn(current_position)
        past_function_value = self._historic_potential[str(colloid_id)]

        # Compute difference in scaled_distances
        delta = current_function_value - past_function_value

        # Compute the reward
        reward = -100 * delta + abs(current_function_value)

        # Update the historic position
        self._historic_potential[str(colloid_id)] = current_function_value

        return reward
    
    def __call__(self, colloids: List):
        """
        Compute the reward.

        In this case of this task, the observable itself is the gradient of the field
        that the colloid is swimming in. Therefore, the change is simply scaled and
        returned.

        Parameters
        ----------
        colloids : List[Colloid] (n_colloids, )
                List of colloids to be used in the task.

        Returns
        -------
        rewards : List[float] (n_colloids, )
                Rewards for each colloid.
        """
        colloid_indices = self.get_colloid_indices(colloids)

        return np.array(
            [self.compute_colloid_reward(index, colloids) for index in colloid_indices]
        )


In [6]:
class MDObservable(srl.observables.Observable):
    def __init__(
        self,
        potential_fn: callable,
        particle_type: int = 0,
    ):
        """
        Constructor for the find origin task.

        Parameters
        ----------
        source : np.ndarray (default = (0, 0 0))
                Source of the gradient.
        potenial_fn : callable
                Function we are moving along.
        particle_type : int (default=0)

        """
        super().__init__(particle_type=particle_type)
        self.potential_fn = potential_fn

        # Class only attributes
        self._historic_potential = {}
        
    def initialize(self, colloids: List):
        """
        Prepare the task for running.

        Parameters
        ----------
        colloids : List[Colloid]
                List of colloids to be used in the task.

        Returns
        -------
        observable :
                Returns the observable required for the task.
        """
        for item in colloids:
            if item.type == self.particle_type:
                index = item.id
                function_value = self.potential_fn(item.pos)
                self._historic_potential[str(index)] = function_value
                
    def compute_single_observable(self, index: int, colloids: List) -> float:
        """
        Compute the observable for a single colloid.

        Parameters
        ----------
        index : int
                Index of the colloid to compute the observable for.
        colloids : List[Colloid]
                List of colloids in the system.
        """
        colloid_id = colloids[index].id
        # Get the current position of the colloid
        current_position =colloids[index].pos
        current_function_value = self.potential_fn(current_position)
        past_function_value = self._historic_potential[str(colloid_id)]

        # Compute difference in scaled_distances
        delta = current_function_value - past_function_value

        # Update the historic position
        self._historic_potential[str(colloid_id)] = current_function_value

        return 10 * delta
    
    def compute_observable(self, colloids: List):
        """
        Compute the position of the colloid.

        Parameters
        ----------
        colloids : List[Colloid] (n_colloids, )
                List of all colloids in the system.

        Returns
        -------
        observables : List[float] (n_colloids, dimension)
                List of observables, one for each colloid. In this case,
                current field value minus to previous field value.
        """
        reference_ids = self.get_colloid_indices(colloids)

        observables = [
            self.compute_single_observable(index, colloids) for index in reference_ids
        ]

        return np.array(observables).reshape(-1, 1)


In [7]:
def single_well(position: float):
    """
    Scaling function for the task
    """
    return -20 * np.exp(
        -((position[0] - 500)**2 / 5000) - ((position[1] - 500)**2 / 5000)
    )

def double_well(position: float):
    """
    Scaling function for the task
    """
    return -20 * np.exp(
        -((position[0] - 500)**2 / 5000) - ((position[1] - 250)**2 / 5000)
    ) - 20 * np.exp(
        -((position[0] - 500)**2 / 5000) - ((position[1] - 750)**2 / 5000)
    )

def quadruple_well(position: float):
    """
    Scaling function for 4 wells.
    """
    return -20 * np.exp(
        -((position[0] - 250)**2 / 5000) - ((position[1] - 250)**2 / 5000)
    ) - 20 * np.exp(
        -((position[0] - 250)**2 / 5000) - ((position[1] - 750)**2 / 5000)
    ) - 20 * np.exp(
        -((position[0] - 750)**2 / 5000) - ((position[1] - 250)**2 / 5000)
    ) - 20 * np.exp(
        -((position[0] - 750)**2 / 5000) - ((position[1] - 750)**2 / 5000)
    )

# Set the task
task = MinimaDetection(quadruple_well)
observable = MDObservable(quadruple_well)

In [8]:
loss = srl.losses.PolicyGradientLoss(value_function=value_function)

In [9]:
class ActorNet(nn.Module):
    """A simple dense model."""

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=4)(x)
        return x

class CriticNet(nn.Module):
    """A simple dense model."""

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=1)(x)
        return x

In [10]:
actor = srl.networks.FlaxModel(
    flax_model=ActorNet(),
    optimizer=optax.adam(learning_rate=0.001),
    input_shape=(1,),
    sampling_strategy=sampling_strategy,
    exploration_policy=exploration_policy,
)

critic = srl.networks.FlaxModel(
    flax_model=CriticNet(),
    optimizer=optax.adam(learning_rate=0.001),
    input_shape=(1,),
)

In [11]:
translate = Action(force=10.0)
rotate_clockwise = Action(torque=np.array([0.0, 0.0, 10.0]))
rotate_counter_clockwise = Action(torque=np.array([0.0, 0.0, -10.0]))
do_nothing = Action()

actions = {
    "RotateClockwise": rotate_clockwise,
    "Translate": translate,
    "RotateCounterClockwise": rotate_counter_clockwise,
    "DoNothing": do_nothing,
}

In [12]:
protocol = srl.rl_protocols.ActorCritic(
    particle_type=0, 
    actor=actor, 
    critic=critic, 
    task=task, 
    observable=observable, 
    actions=actions
)


In [13]:
rl_trainer = srl.gyms.Gym(
    [protocol],
    loss,
)
rl_trainer.restore_models()

## Model Running

In [14]:
rl_trainer.perform_rl_training(
    system_runner=system_runner,
    n_episodes=n_episodes,
    episode_length=episode_length,
)

Output()

/Users/samueltovey/miniconda3/envs/zincware/lib/python3.10/site-packages/jax/_src/numpy/lax_numpy.py:2365: 
UserWarning: Explicitly requested dtype float64 requested in asarray is not available, and will be truncated to 
dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell 
environment variable. See https://github.com/google/jax#current-gotchas for more.
  start = asarray(start, dtype=computation_dtype)

/Users/samueltovey/miniconda3/envs/zincware/lib/python3.10/site-packages/jax/_src/numpy/lax_numpy.py:2366: 
UserWarning: Explicitly requested dtype float64 requested in asarray is not available, and will be truncated to 
dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell 
environment variable. See https://github.com/google/jax#current-gotchas for more.
  stop = asarray(stop, dtype=computation_dtype)

/Users/samueltovey/miniconda3/envs/zincware/lib/python3.10/site-packages/jax/_src/numpy/lax_numpy.py:2382: 
UserWarning: Explicitly requested dtype float64 requested in astype is not available, and will be truncated to 
dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell 
environment variable. See https://github.com/google/jax#current-gotchas for more.
  step = step.astype(computation_dtype)

/Users/samueltovey/miniconda3/envs/zincware/lib/python3.10/site-packages/jax/_src/numpy/lax_numpy.py:2391: 
UserWarning: Explicitly requested dtype float64 requested in asarray is not available, and will be truncated to 
dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell 
environment variable. See https://github.com/google/jax#current-gotchas for more.
  delta = asarray(nan if endpoint else stop - start, dtype=computation_dtype)


KeyboardInterrupt



In [15]:
rl_trainer.export_models()